# 00 · Data checks

**Question:** Is the match data complete and correct, and does home advantage look different in 2020-21 even before any modelling?

Run this before any analysis. A model can't fix bad data.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt

from ghost_games.data import load_matches
from ghost_games.outcomes import add_outcomes

In [ ]:
matches = add_outcomes(load_matches("../data/raw"))
print(matches.shape)
matches.head()

## 1. Sanity checks
Each `assert` stops the notebook if something is wrong, instead of letting bad data reach the model.

In [ ]:
counts = matches.groupby(["league", "season"]).size()
expected = {"D1": 306, "E0": 380, "SP1": 380, "I1": 380, "F1": 380}
# compare each count with the expected number for its league
expected_counts = counts.index.get_level_values("league").map(expected)
odd = counts[counts.to_numpy() != expected_counts.to_numpy()]
print("League-seasons with an unusual match count:")
print(odd)

assert not matches.duplicated(["league", "date", "home_team", "away_team"]).any()
assert (matches[["home_goals", "away_goals"]] >= 0).all().all()
assert matches["date"].between("2015-07-01", "2023-07-01").all()

## 2. First look: home advantage per season
Mean home goal difference. 0 would mean no home advantage.

In [ ]:
by_season = matches.pivot_table(index="season", columns="league", values="goal_diff", aggfunc="mean")
by_season.style.format("{:+.2f}").background_gradient(cmap="RdYlGn", axis=None)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for league in by_season.columns:
    ax.plot(by_season.index, by_season[league], marker="o", label=league)
ax.axvspan("2019-20", "2020-21", color="grey", alpha=0.15, label="COVID seasons")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Home goal advantage per season")
ax.set_xlabel("Season")
ax.set_ylabel("Mean home goal difference")
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()
# Takeaway: TODO, write one sentence after you see the chart

## 3. Your turn
Before reading on, write down:
1. Which league lost the most home advantage in 2020-21?
2. Did it come back in 2021-22?
3. Can a season-level average answer *why*? (Hint: what else changed that season?)